In [11]:
import json
import re
import sqlite3
import os

In [5]:
topic = "Linux"

In [4]:
def slugify_title(title):
    return re.sub(r"[^a-z0-9]+", "_", title.lower()).strip("_")


def get_connection(db_path="../quiz_outlines.db"):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


In [18]:
def get_outline(conn, article_title):
    row = conn.execute(
        "SELECT article_id, outline_json FROM outlines WHERE article_title = ?",
        (article_title,),
    ).fetchone()
    if row is None:
        return None
    return {
        "article_id": row["article_id"],
        "outline": json.loads(row["outline_json"])
    }


In [19]:
conn = get_connection()
outline = get_outline(conn, topic)

In [20]:
outline

{'article_id': 'linux',
 'outline': {'article_title': 'Linux',
  'sections': [{'breadcrumb': 'Introduction',
    'preview': 'Linux ( LIN-uuks) is a family of free and open-source software Unix-like operating systems based on the Linux kernel, which was first released on 17...',
    'token_count': 412},
   {'breadcrumb': 'Overview',
    'preview': 'The Linux kernel was created by Linus Torvalds, following the lack of a working kernel for GNU, a Unix-compatible operating system made entirely of...',
    'token_count': 590},
   {'breadcrumb': 'History > Precursors',
    'preview': "The Unix operating system was conceived of and implemented in 1969, at AT&T's Bell Labs in the United States, by Ken Thompson, Dennis Ritchie,...",
    'token_count': 537},
   {'breadcrumb': 'History > Creation',
    'preview': 'While attending the University of Helsinki in the fall of 1990, Torvalds enrolled in a Unix course. The course used a MicroVAX minicomputer running...',
    'token_count': 442},
   {'br

In [12]:
PLANNER_SYSTEM_PROMPT = """
You are an assessment planner.

Your task is to design a blueprint for a quiz using only the article outline that you are given.

The outline contains:
- hierarchical section names (breadcrumbs)
- a short preview of each section
- the approximate size of each section

Do NOT generate quiz questions.
Do NOT retrieve information.
Do NOT invent facts that are not implied by the outline.

Your job is only to decide:

1. Which sections should contribute questions.
2. How many questions should come from each section.
3. What difficulty each section should contribute.
4. Why each section was selected.

When creating the blueprint:

- Prefer broad coverage over concentrating questions in a single section.
- Ensure every selected section appears relevant to the user's request.
- Avoid selecting sections that appear too small or too narrow unless they are specifically relevant.
- Large sections may receive multiple questions.
- Introductory sections should usually receive fewer questions than substantive sections.
- If the user requests an easier quiz, favor foundational sections.
- If the user requests a harder quiz, favor advanced or specialized sections.
- The total number of planned questions MUST equal the requested number.

Return ONLY valid JSON.
"""

In [22]:
PLANNER_PROMPT = f"""
User request:

Generate a medium-difficulty quiz about Linux.

Number of questions:
10

Article outline:

{outline}
"""

In [23]:
from google import genai
from google.genai import types

In [24]:
gemini_client = genai.Client()

In [25]:
response = gemini_client.models.generate_content(model="gemini-3.5-flash", config=types.GenerateContentConfig(system_instruction=PLANNER_SYSTEM_PROMPT), contents=PLANNER_PROMPT)

In [34]:
blueprint = response.text.strip().replace("```json", "").replace("```", "").strip()
blueprint = json.loads(blueprint).get("blueprint")
blueprint

[{'breadcrumb': 'Introduction',
  'questions': 1,
  'difficulty': 'easy',
  'reason': 'Provides an essential foundational question regarding the definition and baseline timeline of the Linux operating system family.'},
 {'breadcrumb': 'Overview',
  'questions': 1,
  'difficulty': 'medium',
  'reason': 'A substantial section that explores the relationship between the Linux kernel and the GNU project, which is core to understanding Linux operating systems.'},
 {'breadcrumb': 'History > Precursors',
  'questions': 1,
  'difficulty': 'medium',
  'reason': 'Covers the historical roots of Unix at Bell Labs, which is crucial for testing medium-difficulty historical context.'},
 {'breadcrumb': 'History > Copyright, trademark, and naming',
  'questions': 1,
  'difficulty': 'medium',
  'reason': 'A large and important section discussing the GPL v2 license and distribution requirements, key to open-source governance.'},
 {'breadcrumb': 'Usage > Market share and uptake',
  'questions': 1,
  'diffi

In [35]:
from qdrant_client import QdrantClient, models

client = QdrantClient(url="http://localhost:6333")

res = client.scroll(
    collection_name="Quiz-App-Dev-Collection",
    scroll_filter=models.Filter(
        must=[
            models.FieldCondition(
                key="section_title",
                match=models.MatchValue(value="Overview"),
            ),
        ]
    ),
)


In [36]:
res

([Record(id=1, payload={'article_title': 'Linux', 'section_title': 'Overview', 'source_url': 'https://en.wikipedia.org/wiki/Linux#Overview', 'raw_text': "The Linux kernel was created by Linus Torvalds, following the lack of a working kernel for GNU, a Unix-compatible operating system made entirely of free software that had been in development since 1983 by the GNU Project, led by Richard Stallman. A working Unix system called Minix was later released but its license was not entirely free at the time and it was made for education purposes. The first entirely free Unix for personal computers, 386BSD, did not appear until 1992, by which time Torvalds had already built and publicly released the first version of the Linux kernel on the Internet. Like GNU and 386BSD, Linux did not have any Unix code, being a fresh re-implementation, and therefore avoided legal issues from AT&T. Linux distributions became popular in the 1990s and made Unix technologies accessible to home users on personal com